# Multilingual Topic Modelling — Full Pipeline Demo

End-to-end walkthrough using the `multilingual_topic` library on synthetic data.

**Steps:**
1. Generate dummy multilingual corpus
2. Preprocess text
3. Machine translation (non-English → English)
4. Layer 1 — global topic model
5. Layer 2 — per-theme topic model + SetFit refinement
6. SetFit cross-validation and evaluation
7. Analysis and visualisation

In [ ]:
import pandas as pd
import numpy as np
from multilingual_topic import (
    TextPreprocessor,
    SentenceRepresentation,
    TopicModel,
    train_setfit,
    cross_validate_setfit,
    Evaluation,
    ClassifierValidationHelper,
    volume_over_time,
    Heatmap,
    top_n_distribution,
)
from multilingual_topic.setfit import ClassifierTestingHelper
from datasets import Dataset

## Step 1 — Generate dummy multilingual corpus

In [ ]:
np.random.seed(42)
n = 500

# Simulate posts in multiple languages about two narrative events
templates = {
    'en': [
        'Sweden faces international criticism over religious freedoms',
        'Child welfare system under scrutiny in Muslim communities',
        'Diplomatic tensions rise following protest events in Stockholm',
        'Swedish government responds to disinformation campaign online',
        'International media coverage of Sweden grows significantly',
    ],
    'ar': [
        'السويد تواجه انتقادات دولية بشأن حرية الدين',
        'نظام رعاية الأطفال تحت المجهر في المجتمعات المسلمة',
        'توترات دبلوماسية تتصاعد في أعقاب احتجاجات ستوكهولم',
        'الحكومة السويدية ترد على حملة التضليل الإعلامي',
        'تنامي التغطية الإعلامية الدولية للشأن السويدي',
    ],
    'tr': [
        'İsveç dini özgürlükler konusunda uluslararası eleştiriyle karşılaşıyor',
        'Müslüman topluluklarda çocuk refahı sistemi inceleme altında',
        'Stockholm protestoları sonrası diplomatik gerilimler yükseliyor',
    ],
}

rows = []
for _ in range(n):
    lang = np.random.choice(['en', 'ar', 'tr'], p=[0.5, 0.35, 0.15])
    text = np.random.choice(templates[lang])
    rows.append({
        'text': text,
        'language': lang,
        'country': np.random.choice(['Egypt', 'Turkey', 'Pakistan', 'Indonesia', 'Saudi Arabia']),
        'platform': np.random.choice(['Twitter', 'Facebook', 'Telegram']),
        'date': pd.Timestamp('2022-01-01') + pd.Timedelta(days=int(np.random.randint(0, 600))),
    })

df = pd.DataFrame(rows)
print(f'{len(df)} posts | languages: {df.language.value_counts().to_dict()}')
df.head()

## Step 2 — Preprocessing

In [ ]:
pp = TextPreprocessor(
    apply_remove_links=True,
    apply_remove_mentions=True,
    apply_remove_hashtags=True,
    apply_remove_emojis=True,
)
df['text_clean'] = df['text'].apply(pp.preprocess)
print('Preprocessing complete.')
df[['text', 'text_clean']].head(3)

## Step 3 — Machine translation

In production, non-English content is translated using mBART (many-to-many) or Helsinki-NLP models.
Here we simulate by tagging non-English rows — replace with `translate_iterator` for real use.

In [ ]:
# Simulate: in practice use ManyToManyTranslator + translate_iterator
# from multilingual_topic import ManyToManyTranslator, translate_iterator
#
# translator = ManyToManyTranslator('mbart', model, tokenizer, max_length=512, use_gpu=False)
# translated = list(translate_iterator(
#     iter_df_as_dict(df[df.language != 'en']),
#     batch_size=16, translator=translator,
#     text_col='text_clean', src_lang='ar_AR', tgt_lang='en_XX',
#     translated_col='text_en', translated_by_col='translator'
# ))

# For demo: use original text for English, mark others as needing translation
df['text_en'] = df.apply(
    lambda r: r['text_clean'] if r['language'] == 'en' else f'[translated] {r["text_clean"]}',
    axis=1
)
print(f'English: {(df.language == "en").sum()} | Translated: {(df.language != "en").sum()}')

## Step 4 — Layer 1: Global topic model

Embed all documents and run BERTopic across the full corpus to surface broad themes.

In [ ]:
# Embedding
embedder = SentenceRepresentation(
    model_name='sentence-transformers/all-mpnet-base-v2',
    data=df,
    text_col='text_en',
    file_name='layer1_embeddings',
)
embedder.extract_embeddings(batch_size=32)
print(f'Embeddings shape: {embedder.embeddings.shape}')

In [ ]:
# Topic model
layer1 = TopicModel(data=df, text_col='text_en')
topics, probs = layer1.fit(embedder.embeddings)
df['topic_l1'] = topics
print('Layer 1 topics:', df['topic_l1'].value_counts().head(8).to_dict())

## Step 5 — Layer 2: Per-theme topic model + SetFit refinement

For each broad theme from Layer 1, run a finer BERTopic model.
Then use SetFit to refine cluster boundaries with positive/negative examples.

In [ ]:
# Pick a theme to drill into (e.g. topic 0)
theme_df = df[df['topic_l1'] == 0].copy().reset_index(drop=True)
print(f'Theme 0 contains {len(theme_df)} documents')

# Prepare SetFit training data: label relevant vs irrelevant within this cluster
# In practice, you annotate a sample; here we simulate
theme_df['label'] = np.where(
    theme_df['text_en'].str.contains('diplomatic|criticism|government', case=False),
    'relevant', 'irrelevant'
)
print(theme_df['label'].value_counts().to_dict())

In [ ]:
# Build SetFit training dataset (few-shot: 8 examples per class)
helper = ClassifierValidationHelper(
    data=theme_df,
    theme_col='label',
    theme='relevant',
    src_text='text_en',
    text_col='text_en',
    label_col='label',
    prediction_col='pred',
    topic_col='topic_l1',
    sample_size=8,
)
cleaned = helper.prepare_data()
train_df, valid_df = helper.split_data(cleaned)
train_ds, valid_ds = helper.convert_split_to_dataset(train_df, valid_df, sample_size=8)
print(f'Train: {len(train_ds)} | Valid: {len(valid_ds)}')

In [ ]:
# Train SetFit — steers embedding space for the next BERTopic layer
setfit_model = train_setfit(
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    model_name='sentence-transformers/all-mpnet-base-v2',
    num_epochs=1,
    batch_size=8,
)
print('SetFit model trained.')

## Step 6 — SetFit evaluation

In [ ]:
y_pred = setfit_model(valid_ds['text'])
y_true = valid_ds['label']

ev = Evaluation(y_true, y_pred)
ev.report()
fig = ev.confusion_matrix()
fig.show()

In [ ]:
# Cross-validation on the full theme subset
# (requires enough labelled data; shown here for reference)
#
# results = cross_validate_setfit(
#     theme_df, text_col='text_en', label_col='label',
#     model_name='sentence-transformers/all-mpnet-base-v2',
#     n_folds=5, batch_size=8, num_epochs=1,
# )

## Step 7 — Analysis and visualisation

In [ ]:
# Volume over time by country
fig = volume_over_time(df, date_col='date', group_col='country', top_n=5, title='Post volume by country')
fig.show()

In [ ]:
# Country × Layer 1 topic heatmap
hm = Heatmap(df, row_col='country', col_col='topic_l1')
fig = hm.plot(title='Country × Topic distribution')
fig.show()

In [ ]:
# Platform distribution
fig = top_n_distribution(df, col='platform', title='Posts by platform')
fig.show()